# Teacher Network Analysis

Analysis of the generated teacher network activity.

**Sections:**
1. Odourant Analysis - network responses to different odourant patterns vs baseline

In [ ]:
import shutil
import numpy as np
import torch
import toml
import zarr
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from IPython.display import display as ipy_display

from connectome_snns.utils.reproducibility import load_experiment_config
from connectome_snns.dataloaders.unsupervised import HomogeneousPoissonSpikeDataLoader
from connectome_snns.dataloaders.odourants import (
    generate_odour_firing_rates,
    generate_baseline_firing_rates,
)
from connectome_snns.network_simulators.conductance_based.simulator import ConductanceLIFNetwork
from connectome_snns.network_simulators.projections import make_frozen_projections
from connectome_snns.configs import SimulationConfig
from connectome_snns.configs.conductance_based import RecurrentLayerConfig, FeedforwardLayerConfig
from connectome_snns.configs.odours import OdourInputConfig
from connectome_snns.visualization import use_project_style, HIGHLIGHT_COLOR, BASELINE_BAR_COLOR
from connectome_snns.visualization.firing_statistics import plot_cross_correlation_scatter
from connectome_snns.visualization.neuronal_dynamics import plot_synaptic_conductances
from connectome_snns.visualization.odours import plot_input_firing_rate_histogram
from connectome_snns.visualization.dashboards import (
    create_activity_dashboard,
    create_assembly_activity_dashboard,
)

use_project_style()

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# === CONFIGURE PATHS ===
# Point these to your experiment output directory
experiment_dir = load_experiment_config("experiment.toml")["output_dir"]

In [ ]:
# Load parameters
params_file = experiment_dir / "parameters.toml"
with open(params_file, "r") as f:
    data = toml.load(f)

simulation = SimulationConfig(**data["simulation"])
recurrent = RecurrentLayerConfig(**data["recurrent"])
feedforward = FeedforwardLayerConfig(**data["feedforward"])

# Parse odour configs
odours_data = data["odours"].copy()
ou_tau = odours_data.pop("tau")
ou_temperature = odours_data.pop("temperature")
ou_sigma = odours_data.pop("sigma")
odours = {name: OdourInputConfig(**config) for name, config in odours_data.items()}

# Load network structure
input_dir = experiment_dir / "results"
network_structure = np.load(input_dir / "network_structure.npz")

weights = network_structure["recurrent_weights"]
feedforward_weights = network_structure["feedforward_weights"]
cell_type_indices = network_structure["cell_type_indices"]
input_source_indices = network_structure["feedforward_cell_type_indices"]
assembly_ids = network_structure["assembly_ids"]

n_assemblies = len(np.unique(assembly_ids[assembly_ids >= 0]))
print(
    f"Loaded network with {len(cell_type_indices)} neurons ({n_assemblies} assemblies)"
)

# Set seed
if simulation.seed is not None:
    np.random.seed(simulation.seed)
    torch.manual_seed(simulation.seed)

## 1. Odourant Analysis

Compare network responses to odourant-modulated inputs vs baseline (homogeneous Poisson). We test:
- **Odourant 1 vs Baseline**: Does the network respond differently to structured input?
- **Odourant 1 vs Odourant 1 (repeat)**: How much variability comes from Poisson noise alone?

In [ ]:
# Generate odour-modulated firing rate patterns
input_firing_rates_odour = generate_odour_firing_rates(
    feedforward_weights=feedforward_weights,
    input_source_indices=input_source_indices,
    cell_type_indices=cell_type_indices,
    assembly_ids=assembly_ids,
    target_cell_type_idx=0,
    cell_type_names=feedforward.cell_types.names,
    odour_configs={name: cfg.to_dict() for name, cfg in odours.items()},
)

num_odours = len(np.unique(assembly_ids[assembly_ids >= 0]))
print(f"Generated {num_odours} odour patterns, shape: {input_firing_rates_odour.shape}")

# Generate baseline firing rates
baseline_firing_rates = generate_baseline_firing_rates(
    n_input_neurons=feedforward_weights.shape[0],
    input_source_indices=input_source_indices,
    cell_type_names=feedforward.cell_types.names,
    odour_configs={name: cfg.to_dict() for name, cfg in odours.items()},
)
print(f"Baseline firing rates: {baseline_firing_rates.shape}")

In [ ]:
# Input firing rate histogram for odourant 1
fig = plot_input_firing_rate_histogram(
    firing_rates=input_firing_rates_odour,
    bins=30,
)
plt.show()

In [ ]:
# Initialize network
rec_cell_params = recurrent.get_cell_params()
ff_cell_params = feedforward.get_cell_params()

rec_projections, ff_projections = make_frozen_projections(
    rec_weights=weights.astype(np.float32),
    ff_weights=feedforward_weights.astype(np.float32),
    cell_type_indices=cell_type_indices,
    ff_cell_type_indices=input_source_indices,
    cell_type_names=[c["name"] for c in rec_cell_params],
    ff_cell_type_names=[c["name"] for c in ff_cell_params],
)

model = ConductanceLIFNetwork(
    dt=simulation.dt,
    rec_projections=rec_projections,
    ff_projections=ff_projections,
    cell_type_indices=cell_type_indices,
    cell_type_indices_FF=input_source_indices,
    cell_params=rec_cell_params,
    cell_params_FF=ff_cell_params,
    synapse_params=recurrent.get_synapse_params(),
    synapse_params_FF=feedforward.get_synapse_params(),
    surrgrad_scale=1.0,
    batch_size=1,
    track_variables=True,
).to(device)

print("Network initialized")

In [ ]:
# Generate network responses for 3 conditions:
# Odourant 1, Odourant 1 (repeat with different noise seed), Baseline

chunk_size = int(simulation.chunk_size)
dt = simulation.dt
num_chunks = simulation.plot_size if hasattr(simulation, "plot_size") else 10

pattern_cache_path = input_dir / "viz_pattern_spikes.npz"

if pattern_cache_path.exists():
    print(f"Loading pattern spikes from cache: {pattern_cache_path}")
    cached = np.load(pattern_cache_path)
    spikes_odour_1 = cached["spikes_odour_1"]
    spikes_odour_1_repeat = cached["spikes_odour_1_repeat"]
    spikes_baseline = cached["spikes_baseline"]
    print(f"Loaded: {spikes_odour_1.shape}")
else:
    firing_rates_pattern_1 = input_firing_rates_odour[0:1]

    # Odourant 1 with different seed for repeat
    torch.manual_seed(simulation.seed + 1 if simulation.seed is not None else 42)
    np.random.seed((simulation.seed + 1) if simulation.seed is not None else 42)
    firing_rates_pattern_2 = generate_odour_firing_rates(
        feedforward_weights=feedforward_weights,
        input_source_indices=input_source_indices,
        cell_type_indices=cell_type_indices,
        assembly_ids=assembly_ids,
        target_cell_type_idx=0,
        cell_type_names=feedforward.cell_types.names,
        odour_configs={name: cfg.to_dict() for name, cfg in odours.items()},
    )[0:1]

    # Stack all 3 patterns: (3, n_input_neurons)
    all_firing_rates = np.vstack(
        [firing_rates_pattern_1, firing_rates_pattern_2, baseline_firing_rates]
    )

    # Reset seed
    torch.manual_seed(simulation.seed)
    np.random.seed(simulation.seed)

    dataloader = HomogeneousPoissonSpikeDataLoader(
        firing_rates=all_firing_rates,
        chunk_size=chunk_size,
        dt=dt,
        batch_size=1,
        device=device,
    )

    all_spikes_chunks = []
    with torch.inference_mode():
        for chunk_idx, (input_spikes_chunk, pattern_indices) in enumerate(
            tqdm(dataloader, total=num_chunks, desc="Simulating", leave=False)
        ):
            input_spikes_chunk = input_spikes_chunk.to(device)
            batch_size, n_patterns, time_steps, n_inputs = input_spikes_chunk.shape

            chunk_outputs = []
            for pattern_idx in range(n_patterns):
                if chunk_idx == 0:
                    model.reset_state()
                input_pattern = input_spikes_chunk[:, pattern_idx, :, :]
                outputs = model.forward(input_spikes=input_pattern)
                chunk_outputs.append(outputs["spikes"].cpu())

            all_spikes_chunks.append(torch.stack(chunk_outputs, dim=1))
            if chunk_idx + 1 >= num_chunks:
                break

    # Concatenate: (batch=1, n_patterns=3, total_time, n_neurons)
    spikes_all_patterns = torch.cat(all_spikes_chunks, dim=2)

    spikes_odour_1 = spikes_all_patterns[:, 0, :, :].numpy()
    spikes_odour_1_repeat = spikes_all_patterns[:, 1, :, :].numpy()
    spikes_baseline = spikes_all_patterns[:, 2, :, :].numpy()

    np.savez_compressed(
        pattern_cache_path,
        spikes_odour_1=spikes_odour_1,
        spikes_odour_1_repeat=spikes_odour_1_repeat,
        spikes_baseline=spikes_baseline,
    )
    print(f"Saved pattern spikes to {pattern_cache_path}")

print(f"Simulation complete: {spikes_odour_1.shape}")

In [ ]:
# Odourant 1 vs Baseline
fig = plot_cross_correlation_scatter(
    spike_trains_trial1=spikes_odour_1,
    spike_trains_trial2=spikes_baseline,
    window_size=10.0,
    dt=dt,
    title="Network Activity: Odourant 1 vs Baseline",
    x_label="Odourant 1 Firing Rate (Hz)",
    y_label="Baseline Firing Rate (Hz)",
    max_rate=50,
)
plt.show()

In [ ]:
# Odourant 1 vs Odourant 1 (different Poisson noise)
fig = plot_cross_correlation_scatter(
    spike_trains_trial1=spikes_odour_1,
    spike_trains_trial2=spikes_odour_1_repeat,
    window_size=10.0,
    dt=dt,
    title="Network Activity: Same Odourant, Different Noise",
    x_label="Odourant 1 Firing Rate (Hz)",
    y_label="Odourant 1 Firing Rate (noise variant) (Hz)",
    max_rate=50,
)
plt.show()

In [ ]:
# Schematic illustration of input coding: baseline vs odourant stimulus
# Shows how one odourant elevates its assembly's input while the rest are
# slightly depressed to keep the mean firing rate constant.

n_odourants = 20
baseline_rate = 6.0
active_rate = 15.0
depressed_rate = (n_odourants * baseline_rate - active_rate) / (n_odourants - 1)

x = np.arange(n_odourants)
odourant_labels = [str(i + 1) for i in range(n_odourants)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

# Left: baseline — all 20 odourants at equal rate
axes[0].bar(x, baseline_rate, color=BASELINE_BAR_COLOR, alpha=0.8, width=0.7)
axes[0].axhline(baseline_rate, color="k", linestyle="--", linewidth=0.8, alpha=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(odourant_labels, fontsize=8)
axes[0].set_xlabel("Odourant")
axes[0].set_ylabel("Input firing rate (Hz)")
axes[0].set_title("Baseline")
axes[0].set_ylim(0, active_rate * 1.15)

# Right: odourant 1 active — elevated, rest depressed to preserve mean
heights = np.full(n_odourants, depressed_rate)
heights[0] = active_rate
bar_colors = [
    HIGHLIGHT_COLOR if i == 0 else BASELINE_BAR_COLOR for i in range(n_odourants)
]
axes[1].bar(x, heights, color=bar_colors, alpha=0.8, width=0.7)
axes[1].axhline(
    baseline_rate,
    color="k",
    linestyle="--",
    linewidth=0.8,
    alpha=0.5,
    label=f"Mean = {baseline_rate:.0f} Hz",
)
axes[1].set_xticks(x)
axes[1].set_xticklabels(odourant_labels, fontsize=8)
axes[1].set_xlabel("Odourant")
axes[1].set_title("Odourant 1")
axes[1].legend(fontsize=8, loc="upper right")

fig.suptitle("Input patterns: constructed odourant firing rates", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Baseline rate: {baseline_rate:.1f} Hz")
print(f"Active rate:   {active_rate:.1f} Hz")
print(
    f"Depressed rate: {depressed_rate:.3f} Hz  (mean preserved at {baseline_rate:.1f} Hz)"
)

## Activity Dashboard

Regenerates the activity and assembly dashboards from the saved zarr data.
Re-runs the model on the stored feedforward inputs to recover voltages, currents, and conductances.

In [ ]:
plot_size = int(simulation.plot_size)
chunk_size_val = int(simulation.chunk_size)
n_plot_steps = plot_size * chunk_size_val

# Load spikes and OU weights from zarr (saved by the training script)
zarr_root = zarr.open_group(input_dir / "spike_data.zarr", mode="r")
viz_input_spikes = zarr_root["input_spikes"][0:1, :n_plot_steps, :].astype(np.int32)
viz_output_spikes = zarr_root["output_spikes"][0:1, :n_plot_steps, :].astype(np.int32)
viz_ou_weights = zarr_root["weights"][0:1, :n_plot_steps, :].astype(np.float32)

# Cache tracked variables (voltages, currents, conductances) to avoid re-simulating
cache_path = input_dir / "viz_tracked_vars.zarr"
_CACHE_KEYS = {
    "voltages",
    "currents",
    "currents_FF",
    "currents_leak",
    "conductances",
    "conductances_FF",
}

cache_valid = False
if Path(cache_path).exists():
    try:
        _cache_check = zarr.open_group(str(cache_path), mode="r")
        cache_valid = _CACHE_KEYS.issubset(set(_cache_check.keys()))
    except Exception:
        pass
    if not cache_valid:
        print("Cache incomplete or corrupt — removing and recomputing.")
        shutil.rmtree(cache_path)

if cache_valid:
    print(f"Loading tracked variables from cache: {cache_path}")
    cache = zarr.open_group(str(cache_path), mode="r")
    viz_voltages = cache["voltages"][:]
    viz_currents = cache["currents"][:]
    viz_currents_FF = cache["currents_FF"][:]
    viz_currents_leak = cache["currents_leak"][:]
    viz_conductances = cache["conductances"][:]
    viz_conductances_FF = cache["conductances_FF"][:]
    print("Done.")
else:
    print(f"Running model with variable tracking ({plot_size} chunks)...")
    model.reset_state(batch_size=1)
    all_voltages, all_currents, all_currents_leak, all_conductances = [], [], [], []

    with torch.inference_mode():
        for i in tqdm(range(plot_size), desc="Simulating"):
            chunk = (
                torch.from_numpy(
                    viz_input_spikes[
                        :, i * chunk_size_val : (i + 1) * chunk_size_val, :
                    ]
                )
                .float()
                .to(device)
            )
            out = model.forward(input_spikes=chunk)
            all_voltages.append(out["voltages"].cpu().numpy())
            all_currents.append(out["currents"].cpu().numpy())
            all_currents_leak.append(out["currents_leak"].cpu().numpy())
            all_conductances.append(out["conductances"].cpu().numpy())

    viz_voltages = np.concatenate(all_voltages, axis=1).astype(np.float32)
    viz_currents_all = np.concatenate(all_currents, axis=1).astype(np.float32)
    viz_currents_leak = np.concatenate(all_currents_leak, axis=1).astype(np.float32)
    viz_conductances_all = np.concatenate(all_conductances, axis=1).astype(np.float32)

    rec_unified_ids = sorted(set(model.rec_synapse_to_unified.values()))
    ff_unified_ids = sorted(set(model.ff_synapse_to_unified.values()))

    viz_currents = viz_currents_all[..., rec_unified_ids].astype(np.float32)
    viz_currents_FF = viz_currents_all[..., ff_unified_ids].astype(np.float32)
    viz_conductances = viz_conductances_all[..., rec_unified_ids].astype(np.float32)
    viz_conductances_FF = viz_conductances_all[..., ff_unified_ids].astype(np.float32)

    cache = zarr.open_group(str(cache_path), mode="w")
    cache.create_array("voltages", data=viz_voltages)
    cache.create_array("currents", data=viz_currents)
    cache.create_array("currents_FF", data=viz_currents_FF)
    cache.create_array("currents_leak", data=viz_currents_leak)
    cache.create_array("conductances", data=viz_conductances)
    cache.create_array("conductances_FF", data=viz_conductances_FF)
    print(f"Saved tracked variables to {cache_path}")

print(f"Voltages: {viz_voltages.shape}")
print(f"Currents (rec): {viz_currents.shape}")
print(f"Conductances (rec): {viz_conductances.shape}")

In [ ]:
# Set to an integer to pin a specific neuron index, or None to auto-select
viz_neuron_index = 13

activity_fig = create_activity_dashboard(
    output_spikes=viz_output_spikes,
    input_spikes=viz_input_spikes,
    cell_type_indices=cell_type_indices,
    cell_type_names=recurrent.cell_types.names,
    dt=dt,
    voltages=viz_voltages,
    neuron_types=cell_type_indices,
    neuron_params=recurrent.get_neuron_params_for_plotting(),
    recurrent_currents=viz_currents,
    feedforward_currents=viz_currents_FF,
    leak_currents=viz_currents_leak,
    recurrent_conductances=viz_conductances,
    feedforward_conductances=viz_conductances_FF,
    input_cell_type_names=feedforward.cell_types.names,
    recurrent_synapse_names=recurrent.get_synapse_names(),
    feedforward_synapse_names=feedforward.get_synapse_names(),
    window_size=50.0,
    n_neurons_plot=20,
    fraction=1.0,
    random_seed=42,
    assembly_ids=assembly_ids,
    neuron_idx=viz_neuron_index,
)
plt.show()

In [ ]:
activity_raster_fig = create_activity_dashboard(
    output_spikes=viz_output_spikes,
    input_spikes=viz_input_spikes,
    cell_type_indices=cell_type_indices,
    cell_type_names=recurrent.cell_types.names,
    dt=dt,
    voltages=viz_voltages,
    neuron_types=cell_type_indices,
    neuron_params=recurrent.get_neuron_params_for_plotting(),
    recurrent_currents=viz_currents,
    feedforward_currents=viz_currents_FF,
    leak_currents=viz_currents_leak,
    recurrent_conductances=viz_conductances,
    feedforward_conductances=viz_conductances_FF,
    input_cell_type_names=feedforward.cell_types.names,
    recurrent_synapse_names=recurrent.get_synapse_names(),
    feedforward_synapse_names=feedforward.get_synapse_names(),
    window_size=50.0,
    n_neurons_plot=20,
    fraction=1.0,
    random_seed=42,
    assembly_ids=assembly_ids,
    neuron_idx=viz_neuron_index,
    panels="raster",
)
plt.show()

In [ ]:
n_5s = int(5.0 / (dt * 1e-3))

activity_neuron_fig = create_activity_dashboard(
    output_spikes=viz_output_spikes[:, :n_5s, :],
    input_spikes=viz_input_spikes[:, :n_5s, :],
    cell_type_indices=cell_type_indices,
    cell_type_names=recurrent.cell_types.names,
    dt=dt,
    voltages=viz_voltages[:, :n_5s, :],
    neuron_types=cell_type_indices,
    neuron_params=recurrent.get_neuron_params_for_plotting(),
    recurrent_currents=viz_currents[:, :n_5s, :, :],
    feedforward_currents=viz_currents_FF[:, :n_5s, :, :],
    leak_currents=viz_currents_leak[:, :n_5s, :],
    recurrent_conductances=viz_conductances[:, :n_5s, :, :, :],
    feedforward_conductances=viz_conductances_FF[:, :n_5s, :, :, :],
    input_cell_type_names=feedforward.cell_types.names,
    recurrent_synapse_names=recurrent.get_synapse_names(),
    feedforward_synapse_names=feedforward.get_synapse_names(),
    window_size=50.0,
    n_neurons_plot=20,
    fraction=1.0,
    random_seed=42,
    assembly_ids=assembly_ids,
    neuron_idx=viz_neuron_index,
    panels="neuron",
)
plt.show()

In [ ]:
# Change this to use a different trial (batch index)
viz_assembly_trial = 2

_zarr_root = zarr.open_group(input_dir / "spike_data.zarr", mode="r")
_n_batches = _zarr_root["output_spikes"].shape[0]
if viz_assembly_trial >= _n_batches:
    raise ValueError(
        f"viz_assembly_trial={viz_assembly_trial} but zarr only has {_n_batches} batches"
    )

assembly_output_spikes = _zarr_root["output_spikes"][
    viz_assembly_trial : viz_assembly_trial + 1, :n_plot_steps, :
].astype(np.int32)
assembly_ou_weights = _zarr_root["weights"][
    viz_assembly_trial : viz_assembly_trial + 1, :n_plot_steps, :
].astype(np.float32)
print(f"Trial {viz_assembly_trial} — OU weights t=0: {assembly_ou_weights[0, 0, :4]}")

plt.close("all")
assembly_fig = create_assembly_activity_dashboard(
    output_spikes=assembly_output_spikes,
    ou_process_weights=assembly_ou_weights,
    cell_type_indices=cell_type_indices,
    assembly_ids=assembly_ids,
    dt=dt,
    excitatory_idx=0,
)
assembly_fig.suptitle("Ornstein-Uhlenbeck Input Dynamics", y=1.01)
ipy_display(assembly_fig)
plt.close(assembly_fig)

## Feedforward vs Recurrent Drive

Ratio of total excitatory synaptic drive from feedforward inputs vs recurrent connections.

In [ ]:
# Ratio of total synaptic drive: recurrent excitatory vs feedforward
# Drive proxy = sum over source neurons of (spike_count * total outgoing weight)
# Uses full zarr dataset, not just the visualization window.

# Identify excitatory recurrent neurons
_exc_mask = np.array(
    [
        recurrent.cell_types.names[ct].lower().startswith("excit")
        for ct in cell_type_indices
    ]
)

# Accumulate spike counts over full dataset in chunks (memory-efficient)
_zarr_root = zarr.open_group(input_dir / "spike_data.zarr", mode="r")
_n_batches, _n_total_steps, _n_neurons = _zarr_root["output_spikes"].shape
_n_inputs = _zarr_root["input_spikes"].shape[2]

_output_spike_counts = np.zeros(_n_neurons, dtype=np.int64)
_input_spike_counts = np.zeros(_n_inputs, dtype=np.int64)
for _s in range(0, _n_total_steps, chunk_size_val):
    _output_spike_counts += _zarr_root["output_spikes"][
        :, _s : _s + chunk_size_val, :
    ].sum(axis=(0, 1))
    _input_spike_counts += _zarr_root["input_spikes"][
        :, _s : _s + chunk_size_val, :
    ].sum(axis=(0, 1))

# Total outgoing weight from each excitatory neuron (weights is (source, target))
_exc_outgoing_weights = weights[_exc_mask, :].sum(axis=1)  # (n_exc,)
# Total outgoing weight from each feedforward neuron (feedforward_weights is (source, target))
_ff_outgoing_weights = feedforward_weights.sum(axis=1)  # (n_ff,)

_rec_exc_drive = (_output_spike_counts[_exc_mask] * _exc_outgoing_weights).sum()
_ff_drive = (_input_spike_counts * _ff_outgoing_weights).sum()
_total = _rec_exc_drive + _ff_drive

# Bar chart (normalised)
fig, ax = plt.subplots(figsize=(4, 3.5))
labels = ["Feedforward", "Recurrent Excitatory"]
values = [_ff_drive / _total, _rec_exc_drive / _total]

ax.bar(labels, values, color=BASELINE_BAR_COLOR, alpha=0.6, width=0.5)
ax.set_ylabel("Total synaptic drive")
ax.set_title("Feedforward vs Recurrent Excitatory Drive")

plt.tight_layout()
fig.savefig(input_dir / "synaptic_drive.svg", bbox_inches="tight")
plt.show()

print(f"Ratio (rec exc / FF): {_rec_exc_drive / _ff_drive:.3f}")
print(f"Saved to {input_dir / 'synaptic_drive.svg'}")

In [ ]:
# Plot input conductances for 10 randomly selected neurons
rng = np.random.default_rng(42)
_n_neurons = viz_conductances.shape[2]
sample_neuron_ids = sorted(rng.choice(_n_neurons, size=10, replace=False))

# Sum over rise/decay (axis 3) to get total conductances
rec_g = viz_conductances.sum(axis=3)  # (batch, time, neurons, n_rec_syn)
ff_g = viz_conductances_FF.sum(axis=3)  # (batch, time, neurons, n_ff_syn)

# Use first 5 s for readability
n_5s = int(5.0 / (dt * 1e-3))
rec_g_short = rec_g[:, :n_5s, :, :]
ff_g_short = ff_g[:, :n_5s, :, :]

for nid in sample_neuron_ids:
    ct_name = recurrent.cell_types.names[cell_type_indices[nid]]
    fig = plot_synaptic_conductances(
        recurrent_conductances=rec_g_short,
        feedforward_conductances=ff_g_short,
        cell_type_indices=cell_type_indices,
        cell_type_names=recurrent.cell_types.names,
        input_cell_type_names=feedforward.cell_types.names,
        recurrent_synapse_names=recurrent.get_synapse_names(),
        feedforward_synapse_names=feedforward.get_synapse_names(),
        dt=dt,
        neuron_id=nid,
        fraction=1.0,
    )
    fig.suptitle(f"Neuron {nid} ({ct_name})", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Integrated conductance ratios (recurrent vs feedforward, AMPA vs NMDA)
# Build synapse index maps from the config
rec_syn_names = recurrent.get_synapse_names()
ff_syn_names = feedforward.get_synapse_names()

# Recurrent synapse indices (iterate cell types × synapses in order)
rec_syn_idx = {}
idx = 0
for ct_name in recurrent.cell_types.names:
    for syn_name in rec_syn_names[ct_name]:
        rec_syn_idx[(ct_name, syn_name)] = idx
        idx += 1

# Feedforward synapse indices
ff_syn_idx = {}
idx = 0
for ct_name in feedforward.cell_types.names:
    for syn_name in ff_syn_names[ct_name]:
        ff_syn_idx[(ct_name, syn_name)] = idx
        idx += 1

# Integrate conductances over time (sum * dt) for the sampled neurons
dt_s = dt * 1e-3  # convert ms → s
rows = []
for nid in sample_neuron_ids:
    ct_name = recurrent.cell_types.names[cell_type_indices[nid]]
    row = {"neuron": nid, "type": ct_name}

    # Recurrent AMPA & NMDA (from excitatory presynaptic)
    for syn in ["AMPA", "NMDA"]:
        key = ("excitatory", syn) if ("excitatory", syn) in rec_syn_idx else None
        if key:
            g = rec_g[0, :, nid, rec_syn_idx[key]]
            row[f"rec_{syn}"] = g.sum() * dt_s

    # Feedforward AMPA & NMDA
    for syn in ["AMPA", "NMDA"]:
        for ff_ct in feedforward.cell_types.names:
            key = (ff_ct, syn)
            if key in ff_syn_idx:
                g = ff_g[0, :, nid, ff_syn_idx[key]]
                row[f"ff_{syn}"] = g.sum() * dt_s

    rows.append(row)

# Build display table
header = f"{'Neuron':>8s}  {'Type':>12s}  {'Rec AMPA':>10s}  {'Rec NMDA':>10s}  {'FF AMPA':>10s}  {'FF NMDA':>10s}  {'Rec/FF':>8s}  {'NMDA/AMPA':>10s}"
print(header)
print("-" * len(header))
for r in rows:
    rec_total = r.get("rec_AMPA", 0) + r.get("rec_NMDA", 0)
    ff_total = r.get("ff_AMPA", 0) + r.get("ff_NMDA", 0)
    total_ampa = r.get("rec_AMPA", 0) + r.get("ff_AMPA", 0)
    total_nmda = r.get("rec_NMDA", 0) + r.get("ff_NMDA", 0)
    ratio_rec_ff = rec_total / ff_total if ff_total > 0 else float("inf")
    ratio_nmda_ampa = total_nmda / total_ampa if total_ampa > 0 else float("inf")
    print(
        f"{r['neuron']:>8d}  {r['type']:>12s}"
        f"  {r.get('rec_AMPA', 0):>10.2f}  {r.get('rec_NMDA', 0):>10.2f}"
        f"  {r.get('ff_AMPA', 0):>10.2f}  {r.get('ff_NMDA', 0):>10.2f}"
        f"  {ratio_rec_ff:>8.2f}  {ratio_nmda_ampa:>10.2f}"
    )

# Summary
rec_ampa_all = sum(r.get("rec_AMPA", 0) for r in rows)
rec_nmda_all = sum(r.get("rec_NMDA", 0) for r in rows)
ff_ampa_all = sum(r.get("ff_AMPA", 0) for r in rows)
ff_nmda_all = sum(r.get("ff_NMDA", 0) for r in rows)
print(
    f"\nMean Rec/FF ratio:   {(rec_ampa_all + rec_nmda_all) / (ff_ampa_all + ff_nmda_all):.2f}"
)
print(
    f"Mean NMDA/AMPA ratio: {(rec_nmda_all + ff_nmda_all) / (rec_ampa_all + ff_ampa_all):.2f}"
)